Predictive Modeling


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(r"C:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\data\processed\lifetwin_clean_dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (10000, 20)


In [3]:
from sklearn.model_selection import train_test_split

# Get unique users
users = df["user_id"].astype(str).unique().tolist()

# Same random seed used during preprocessing
train_users, temp_users = train_test_split(
    users,
    test_size=0.30,
    random_state=42
)

val_users, test_users = train_test_split(
    temp_users,
    test_size=38,
    random_state=42
)

print("Train users:", len(train_users))
print("Validation users:", len(val_users))
print("Test users:", len(test_users))

Train users: 175
Validation users: 37
Test users: 38


In [4]:
train_df = df[df["user_id"].astype(str).isin(train_users)].copy()
val_df = df[df["user_id"].astype(str).isin(val_users)].copy()
test_df = df[df["user_id"].astype(str).isin(test_users)].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7000, 20)
Validation shape: (1480, 20)
Test shape: (1520, 20)


In [5]:
train_set = set(train_users)
val_set = set(val_users)
test_set = set(test_users)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [6]:
import pandas as pd

user_split = pd.DataFrame({
    "user_id": train_users + val_users + test_users,
    "split": (
        ["train"] * len(train_users)
        + ["validation"] * len(val_users)
        + ["test"] * len(test_users)
    )
})

user_split.to_csv(
    "../data/processed/user_split.csv",
    index=False
)

print(user_split["split"].value_counts())

split
train         175
test           38
validation     37
Name: count, dtype: int64


In [7]:
user_split = pd.read_csv(
    "../data/processed/user_split.csv"
)

train_users = user_split.loc[
    user_split["split"] == "train", "user_id"
].astype(str).tolist()

val_users = user_split.loc[
    user_split["split"] == "validation", "user_id"
].astype(str).tolist()

test_users = user_split.loc[
    user_split["split"] == "test", "user_id"
].astype(str).tolist()

In [8]:
train_df = df[df["user_id"].astype(str).isin(train_users)].copy()
val_df = df[df["user_id"].astype(str).isin(val_users)].copy()
test_df = df[df["user_id"].astype(str).isin(test_users)].copy()

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7000, 20)
(1480, 20)
(1520, 20)


In [9]:
behavioral_features = [
    "sleep_hours",
    "sleep_quality",
    "wake_time_consistency",
    "study_work_hours",
    "focus_sessions",
    "screen_time_hours",
    "social_media_hours",
    "notification_interruptions",
    "exercise_minutes",
    "step_count",
    "stress_level",
    "mood_score",
    "energy_level",
    "caffeine_intake",
    "break_minutes"
]

categorical_features = [
    "age_group",
    "day_of_week"
]

target = "productivity_score"

In [10]:
X_train_raw = train_df[behavioral_features + categorical_features]
X_val_raw = val_df[behavioral_features + categorical_features]
X_test_raw = test_df[behavioral_features + categorical_features]

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

print("X_train:", X_train_raw.shape)
print("X_val:", X_val_raw.shape)
print("X_test:", X_test_raw.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (7000, 17)
X_val: (1480, 17)
X_test: (1520, 17)
y_train: (7000,)
y_val: (1480,)
y_test: (1520,)


In [11]:
import joblib

encoder = joblib.load(
    "../models/preprocessing/encoder.pkl"
)

scaler = joblib.load(
    "../models/preprocessing/scaler.pkl"
)

print("Encoder loaded:", type(encoder).__name__)
print("Scaler loaded:", type(scaler).__name__)

Encoder loaded: OneHotEncoder
Scaler loaded: StandardScaler


c:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [12]:
# Encode categorical features
train_cat = encoder.transform(train_df[categorical_features])
val_cat = encoder.transform(val_df[categorical_features])
test_cat = encoder.transform(test_df[categorical_features])

# Scale behavioral features
train_num = scaler.transform(train_df[behavioral_features])
val_num = scaler.transform(val_df[behavioral_features])
test_num = scaler.transform(test_df[behavioral_features])

# Combine numerical + categorical features
X_train = np.hstack([train_num, train_cat])
X_val = np.hstack([val_num, val_cat])
X_test = np.hstack([test_num, test_cat])

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (7000, 25)
X_val: (1480, 25)
X_test: (1520, 25)


In [13]:
baseline_prediction = np.full(
    len(y_val),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_val,
    baseline_prediction
)

baseline_mse = mean_squared_error(
    y_val,
    baseline_prediction
)

baseline_rmse = np.sqrt(baseline_mse)

baseline_r2 = r2_score(
    y_val,
    baseline_prediction
)

print("Baseline Productivity:", round(y_train.mean(), 2))
print("Baseline MAE:", round(baseline_mae, 4))
print("Baseline MSE:", round(baseline_mse, 4))
print("Baseline RMSE:", round(baseline_rmse, 4))
print("Baseline R²:", round(baseline_r2, 4))

Baseline Productivity: 48.81
Baseline MAE: 14.7641
Baseline MSE: 316.8118
Baseline RMSE: 17.7992
Baseline R²: -0.0077


In [14]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_val_pred = linear_model.predict(X_val)

linear_mae = mean_absolute_error(
    y_val,
    linear_val_pred
)

linear_mse = mean_squared_error(
    y_val,
    linear_val_pred
)

linear_rmse = np.sqrt(linear_mse)

linear_r2 = r2_score(
    y_val,
    linear_val_pred
)

print("Linear Regression Results")
print("-------------------------")
print("MAE:", round(linear_mae, 4))
print("MSE:", round(linear_mse, 4))
print("RMSE:", round(linear_rmse, 4))
print("R²:", round(linear_r2, 4))

Linear Regression Results
-------------------------
MAE: 8.7129
MSE: 119.5306
RMSE: 10.933
R²: 0.6198


In [15]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_val_pred = rf_model.predict(X_val)

rf_mae = mean_absolute_error(y_val, rf_val_pred)
rf_mse = mean_squared_error(y_val, rf_val_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_val, rf_val_pred)

print("Random Forest Regression Results")
print("---------------------------------")
print("MAE:", round(rf_mae, 4))
print("MSE:", round(rf_mse, 4))
print("RMSE:", round(rf_rmse, 4))
print("R²:", round(rf_r2, 4))


Random Forest Regression Results
---------------------------------
MAE: 8.2582
MSE: 111.4
RMSE: 10.5546
R²: 0.6457


In [16]:
regression_results = pd.DataFrame({
    "Model": [
        "Mean Baseline",
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        rf_mae
    ],
    "MSE": [
        baseline_mse,
        linear_mse,
        rf_mse
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        rf_rmse
    ],
    "R2": [
        baseline_r2,
        linear_r2,
        rf_r2
    ]
})

regression_results

,Model,MAE,MSE,RMSE,R2
0,Mean Baseline,14.764110,316.811839,17.799209,-0.007663
1,Linear Regression,8.712899,119.530611,10.933006,0.619817
2,Random Forest,8.258224,111.400026,10.554621,0.645677


In [17]:
regression_results.sort_values(
    by="RMSE",
    ascending=True
)

,Model,MAE,MSE,RMSE,R2
2,Random Forest,8.258224,111.400026,10.554621,0.645677
1,Linear Regression,8.712899,119.530611,10.933006,0.619817
0,Mean Baseline,14.764110,316.811839,17.799209,-0.007663


In [18]:
from sklearn.model_selection import GroupKFold, cross_validate

In [19]:
groups = train_df["user_id"].astype(str).values

print("Training records:", len(groups))
print("Unique users:", len(np.unique(groups)))

Training records: 7000
Unique users: 175


In [20]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    group_kfold.split(X_train, y_train, groups=groups),
    start=1
):
    print(
        f"Fold {fold}: "
        f"train={len(cv_train_idx)}, "
        f"validation={len(cv_val_idx)}, "
        f"validation users={len(np.unique(groups[cv_val_idx]))}"
    )

Fold 1: train=5600, validation=1400, validation users=35
Fold 2: train=5600, validation=1400, validation users=35
Fold 3: train=5600, validation=1400, validation users=35
Fold 4: train=5600, validation=1400, validation users=35
Fold 5: train=5600, validation=1400, validation users=35


In [21]:
from sklearn.model_selection import cross_validate

cv_results = {}

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups,
        cv=group_kfold,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results[name] = {
        "MAE Mean": -scores["test_MAE"].mean(),
        "MAE Std": scores["test_MAE"].std(),
        "MSE Mean": -scores["test_MSE"].mean(),
        "RMSE Mean": np.sqrt(-scores["test_MSE"].mean()),
        "R2 Mean": scores["test_R2"].mean(),
        "R2 Std": scores["test_R2"].std()
    }

cv_results_df = pd.DataFrame(cv_results).T

cv_results_df

,MAE Mean,MAE Std,MSE Mean,RMSE Mean,R2 Mean,R2 Std
Linear Regression,8.391311,0.341933,114.184270,10.685704,0.647106,0.041732
Random Forest,8.013170,0.290623,103.771173,10.186814,0.679586,0.033040


In [22]:
from sklearn.ensemble import RandomForestRegressor

rf_configs = {
    "RF_200": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "RF_300": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "RF_500": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),

    "RF_depth10": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "RF_depth15": RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}

In [23]:
rf_tuning_results = []

for name, model in rf_configs.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups,
        cv=group_kfold,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "MSE": "neg_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    mean_mae = -scores["test_MAE"].mean()
    mean_mse = -scores["test_MSE"].mean()

    rf_tuning_results.append({
        "Model": name,
        "MAE": mean_mae,
        "RMSE": np.sqrt(mean_mse),
        "R2": scores["test_R2"].mean()
    })

rf_tuning_df = pd.DataFrame(rf_tuning_results)

rf_tuning_df.sort_values(
    by="RMSE",
    ascending=True
)

,Model,MAE,RMSE,R2
2,RF_500,7.995504,10.169593,0.680688
4,RF_depth15,8.010315,10.186264,0.679639
1,RF_300,8.013170,10.186814,0.679586
0,RF_200,8.023969,10.201139,0.678686
3,RF_depth10,8.111626,10.302557,0.672334


In [24]:
final_regressor = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)


In [25]:
final_regressor.fit(
    X_train,
    y_train
)

print("Final Random Forest trained successfully.")

Final Random Forest trained successfully.


In [26]:
final_test_pred = final_regressor.predict(X_test)

test_mae = mean_absolute_error(
    y_test,
    final_test_pred
)

test_mse = mean_squared_error(
    y_test,
    final_test_pred
)

test_rmse = np.sqrt(test_mse)

test_r2 = r2_score(
    y_test,
    final_test_pred
)

print("Final Random Forest — Test Results")
print("-----------------------------------")
print("MAE:", round(test_mae, 4))
print("MSE:", round(test_mse, 4))
print("RMSE:", round(test_rmse, 4))
print("R²:", round(test_r2, 4))

Final Random Forest — Test Results
-----------------------------------
MAE: 8.6369
MSE: 121.2324
RMSE: 11.0106
R²: 0.6545


In [27]:
import joblib
import os

os.makedirs("../models/regression", exist_ok=True)

joblib.dump(
    final_regressor,
    "../models/regression/best_regressor.pkl"
)

print("Final regression model saved successfully.")

Final regression model saved successfully.


In [28]:
print(
    "Model exists:",
    os.path.exists("../models/regression/best_regressor.pkl")
)

Model exists: True


Classification

In [29]:
classification_threshold = y_train.median()

print("Classification threshold:", classification_threshold)

Classification threshold: 49.8


In [30]:
y_train_class = (y_train >= classification_threshold).astype(int)
y_val_class = (y_val >= classification_threshold).astype(int)
y_test_class = (y_test >= classification_threshold).astype(int)

print("Training class distribution:")
print(y_train_class.value_counts())

print("\nValidation class distribution:")
print(y_val_class.value_counts())

print("\nTest class distribution:")
print(y_test_class.value_counts())

Training class distribution:
productivity_score
1    3501
0    3499
Name: count, dtype: int64

Validation class distribution:
productivity_score
0    822
1    658
Name: count, dtype: int64

Test class distribution:
productivity_score
0    918
1    602
Name: count, dtype: int64


In [31]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train_class
)

logistic_val_pred = logistic_model.predict(X_val)
logistic_val_prob = logistic_model.predict_proba(X_val)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

logistic_accuracy = accuracy_score(
    y_val_class,
    logistic_val_pred
)

logistic_precision = precision_score(
    y_val_class,
    logistic_val_pred
)

logistic_recall = recall_score(
    y_val_class,
    logistic_val_pred
)

logistic_f1 = f1_score(
    y_val_class,
    logistic_val_pred
)

logistic_roc_auc = roc_auc_score(
    y_val_class,
    logistic_val_prob
)

logistic_cm = confusion_matrix(
    y_val_class,
    logistic_val_pred
)

print("Logistic Regression — Validation Results")
print("------------------------------------------")
print("Accuracy :", round(logistic_accuracy, 4))
print("Precision:", round(logistic_precision, 4))
print("Recall   :", round(logistic_recall, 4))
print("F1 Score :", round(logistic_f1, 4))
print("ROC-AUC  :", round(logistic_roc_auc, 4))

print("\nConfusion Matrix:")
print(logistic_cm)

Logistic Regression — Validation Results
------------------------------------------
Accuracy : 0.8338
Precision: 0.796
Recall   : 0.8419
F1 Score : 0.8183
ROC-AUC  : 0.9164

Confusion Matrix:
[[680 142]
 [104 554]]


In [33]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm_model.fit(
    X_train,
    y_train_class
)

svm_val_pred = svm_model.predict(X_val)
svm_val_prob = svm_model.predict_proba(X_val)[:, 1]

print("SVM trained successfully.")

c:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM trained successfully.


In [34]:
svm_accuracy = accuracy_score(
    y_val_class,
    svm_val_pred
)

svm_precision = precision_score(
    y_val_class,
    svm_val_pred
)

svm_recall = recall_score(
    y_val_class,
    svm_val_pred
)

svm_f1 = f1_score(
    y_val_class,
    svm_val_pred
)

svm_roc_auc = roc_auc_score(
    y_val_class,
    svm_val_prob
)

svm_cm = confusion_matrix(
    y_val_class,
    svm_val_pred
)

print("SVM — Validation Results")
print("-------------------------")
print("Accuracy :", round(svm_accuracy, 4))
print("Precision:", round(svm_precision, 4))
print("Recall   :", round(svm_recall, 4))
print("F1 Score :", round(svm_f1, 4))
print("ROC-AUC  :", round(svm_roc_auc, 4))

print("\nConfusion Matrix:")
print(svm_cm)

SVM — Validation Results
-------------------------
Accuracy : 0.8507
Precision: 0.8419
Recall   : 0.8176
F1 Score : 0.8296
ROC-AUC  : 0.9213

Confusion Matrix:
[[721 101]
 [120 538]]


In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(
    X_train,
    y_train_class
)

rf_class_val_pred = rf_classifier.predict(X_val)
rf_class_val_prob = rf_classifier.predict_proba(X_val)[:, 1]

print("Random Forest Classifier trained successfully.")

Random Forest Classifier trained successfully.


In [36]:
rf_class_accuracy = accuracy_score(
    y_val_class,
    rf_class_val_pred
)

rf_class_precision = precision_score(
    y_val_class,
    rf_class_val_pred
)

rf_class_recall = recall_score(
    y_val_class,
    rf_class_val_pred
)

rf_class_f1 = f1_score(
    y_val_class,
    rf_class_val_pred
)

rf_class_roc_auc = roc_auc_score(
    y_val_class,
    rf_class_val_prob
)

rf_class_cm = confusion_matrix(
    y_val_class,
    rf_class_val_pred
)

print("Random Forest Classifier — Validation Results")
print("-----------------------------------------------")
print("Accuracy :", round(rf_class_accuracy, 4))
print("Precision:", round(rf_class_precision, 4))
print("Recall   :", round(rf_class_recall, 4))
print("F1 Score :", round(rf_class_f1, 4))
print("ROC-AUC  :", round(rf_class_roc_auc, 4))

print("\nConfusion Matrix:")
print(rf_class_cm)

Random Forest Classifier — Validation Results
-----------------------------------------------
Accuracy : 0.8486
Precision: 0.8478
Recall   : 0.804
F1 Score : 0.8253
ROC-AUC  : 0.9211

Confusion Matrix:
[[727  95]
 [129 529]]


In [37]:
classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

classification_cv_results = []

for name, model in classification_models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train_class,
        groups=groups,
        cv=group_kfold,
        scoring={
            "Accuracy": "accuracy",
            "Precision": "precision",
            "Recall": "recall",
            "F1": "f1",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    classification_cv_results.append({
        "Model": name,
        "Accuracy Mean": scores["test_Accuracy"].mean(),
        "Accuracy Std": scores["test_Accuracy"].std(),
        "Precision Mean": scores["test_Precision"].mean(),
        "Recall Mean": scores["test_Recall"].mean(),
        "F1 Mean": scores["test_F1"].mean(),
        "ROC-AUC Mean": scores["test_ROC_AUC"].mean()
    })

classification_cv_df = pd.DataFrame(
    classification_cv_results
)

classification_cv_df

,Model,Accuracy Mean,Accuracy Std,Precision Mean,Recall Mean,F1 Mean,ROC-AUC Mean
0,Logistic Regression,0.841143,0.017668,0.842906,0.835795,0.839080,0.918392
1,SVM,0.850429,0.017200,0.854160,0.842572,0.848283,0.924479
2,Random Forest,0.843571,0.013263,0.862471,0.813516,0.837172,0.920945


In [38]:
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Final selected classification model
final_classifier = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

# Train only on the training set
final_classifier.fit(X_train, y_train_class)

# Predict on untouched test set
y_test_pred = final_classifier.predict(X_test)
y_test_prob = final_classifier.predict_proba(X_test)[:, 1]

# Test metrics
test_accuracy = accuracy_score(y_test_class, y_test_pred)
test_precision = precision_score(y_test_class, y_test_pred)
test_recall = recall_score(y_test_class, y_test_pred)
test_f1 = f1_score(y_test_class, y_test_pred)
test_roc_auc = roc_auc_score(y_test_class, y_test_prob)

print("Final SVM Test Results")
print("----------------------")
print("Accuracy :", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall   :", round(test_recall, 4))
print("F1 Score :", round(test_f1, 4))
print("ROC-AUC  :", round(test_roc_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_class, y_test_pred))

c:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Final SVM Test Results
----------------------
Accuracy : 0.8632
Precision: 0.8251
Recall   : 0.8306
F1 Score : 0.8278
ROC-AUC  : 0.9346

Confusion Matrix:
[[812 106]
 [102 500]]


In [39]:
import os
import joblib

os.makedirs("../models/classification", exist_ok=True)

joblib.dump(
    final_classifier,
    "../models/classification/best_classifier.pkl"
)

print("Classifier saved successfully.")
print(
    "Model exists:",
    os.path.exists("../models/classification/best_classifier.pkl")
)

Classifier saved successfully.
Model exists: True
